#Severity Estimation Model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# 1. Load Data
df = pd.read_csv('synthetic_crime_severity_real_zones.csv')

# 2. Create Classes (Low/Medium/High) from the Score
# This allows us to calculate F1 Score and Accuracy
bins = [-1, 33, 66, 101]
labels = ['Low', 'Medium', 'High']
df['Severity_Class'] = pd.cut(df['Harm_Score'], bins=bins, labels=labels)

# 3. Preprocessing
le_enc = LabelEncoder()
df['crime_code'] = le_enc.fit_transform(df['crime'])
df['zone_code'] = le_enc.fit_transform(df['Patrol_Zone_DS'])
df['Hour'] = pd.to_datetime(df['time'], format='%H:%M:%S', errors='coerce').dt.hour.fillna(0)

X = df[['crime_code', 'Hour', 'zone_code', 'victim_age']]
y = le_enc.fit_transform(df['Severity_Class']) # Encode Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 4. Compare Models
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(),
    "KNN": KNeighborsClassifier()
}

print(f"{'Model':<20} | {'Accuracy':<10} | {'F1 Score':<10}")
print("-" * 45)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    print(f"{name:<20} | {acc:.4f}     | {f1:.4f}")

Model                | Accuracy   | F1 Score  
---------------------------------------------
Random Forest        | 0.9482     | 0.9483
Gradient Boosting    | 0.9503     | 0.9504
SVM                  | 0.8571     | 0.8538
KNN                  | 0.7598     | 0.7616


mock data for hotspot prediction

In [5]:
import numpy as np
import pandas as pd

# ---- Load your GN list ----
gn_path = "gn_with_closest_police_station.csv"
gn_df = pd.read_csv(gn_path)

gn_df = gn_df.rename(columns={"admin4Pcode": "gn_code", "admin4Name_en": "gn_english_name"})

required_cols = ["gn_code", "GN_population", "distance_to_station_km"]
missing = [c for c in required_cols if c not in gn_df.columns]
if missing:
    raise ValueError(f"Missing columns in GN file: {missing}")

gn_df = gn_df.dropna(subset=["gn_code"]).copy()
gn_df = gn_df.sort_values("gn_code").reset_index(drop=True)

# Create gn_encoded
gn_df["gn_encoded"] = np.arange(len(gn_df), dtype=int)

# ---- Crimes list ----
crimes = ["drugs", "robbery", "theft", "vehical theft", "buglary", "stabbing"]

# ---- Build a realistic base risk signal from real GN features ----
pop = gn_df["GN_population"].fillna(gn_df["GN_population"].median()).astype(float)
dist = gn_df["distance_to_station_km"].fillna(gn_df["distance_to_station_km"].median()).astype(float)

# Normalize helpers
def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-9)

pop_z = zscore(np.log1p(pop))
dist_z = zscore(dist)

base = 0.65 * pop_z + 0.35 * dist_z  # weighted mix

# ---- Crime-specific profiles  ----
crime_profile = {
    "drugs":         {"severity": 1.15, "noise": 0.18},
    "robbery":       {"severity": 1.35, "noise": 0.20},
    "theft":         {"severity": 1.05, "noise": 0.16},
    "vehical theft": {"severity": 1.25, "noise": 0.19},
    "buglary":       {"severity": 1.30, "noise": 0.20},
    "stabbing":      {"severity": 1.45, "noise": 0.22},
}

rng = np.random.default_rng(42)

rows = []
for crime in crimes:
    sev = crime_profile[crime]["severity"]
    noise = crime_profile[crime]["noise"]

    # Create crime-specific signal:
    signal = sev * base + rng.normal(0, noise, size=len(gn_df))

    # Squash into 0..1 like a probability-ish risk score
    risk = 1 / (1 + np.exp(-signal))

    risk = (risk - risk.min()) / (risk.max() - risk.min() + 1e-9)
    risk = np.clip(risk, 0, 1)

    # Build rows
    for i in range(len(gn_df)):
        rows.append({
            "crime_type": crime,
            "gn_encoded": int(gn_df.loc[i, "gn_encoded"]),
            "gn_name": str(gn_df.loc[i, "gn_code"]),
            "risk_score": float(risk[i])
        })

hotspot_predictions = pd.DataFrame(rows)

# ---- Save ----
out_path = "hotspot_predictions.csv"
hotspot_predictions.to_csv(out_path, index=False)

print("Saved:", out_path)
print("Shape:", hotspot_predictions.shape)
print(hotspot_predictions.head(10))


Saved: hotspot_predictions.csv
Shape: (7122, 4)
  crime_type  gn_encoded    gn_name  risk_score
0      drugs           0  LK2103005    0.600258
1      drugs           1  LK2103010    0.579689
2      drugs           2  LK2103015    0.490813
3      drugs           3  LK2103020    0.693728
4      drugs           4  LK2103025    0.109367
5      drugs           5  LK2103030    0.814509
6      drugs           6  LK2103035    0.110677
7      drugs           7  LK2103040    0.379722
8      drugs           8  LK2103045    0.557793
9      drugs           9  LK2103050    0.372394


##Model Training